**Step 1: Install Ultralytics**

This installs the "engine" that handles the training and detection logic.


In [ ]:
!nvidia-smi

In [ ]:
# Install the Ultralytics package
!pip install ultralytics

import os
import yaml
from ultralytics import YOLO
from google.colab import drive

**Step 2: Unzip and Organize Data**

This block extracts your Label Studio files. It assumes your zip has an images folder and a labels folder.

In [ ]:
# 1. Unzip the uploaded file
!unzip -q /content/data.zip -d /content/datasets

# 2. Auto-check the directory structure
dataset_path = "/content/datasets"
print("--- Directory Structure Found ---")
!ls -R {dataset_path} | grep ":$" | sed -e 's/:$//' -e 's/[^-][^\/]*\//--/g' -e 's/^/   /'

# 3. Verify the core folders
import os
folders = os.listdir(dataset_path)
print(f"\nFolders found in /content/datasets: {folders}")

# Check for common Roboflow structure
if 'train' in folders:
    train_images = os.path.join(dataset_path, "train/images")
    if os.path.exists(train_images):
        print(f"✅ Success! Found {len(os.listdir(train_images))} training images.")
    else:
        print("❌ Error: Structure found but 'train/images' is missing.")
else:
    print("❌ Error: No 'train' folder found. Your zip might be structured differently.")

In [ ]:
import os
# This shows 'datasets' folder and its contents, confirming the upload and unzip process worked correctly.
print(os.listdir("/content/datasets"))

**Step 3: Create the Configuration** (`data.yaml`)

YOLO needs a "map" to find your images. This script automatically creates that map based on your Label Studio export.

In [ ]:
# YAML dictionary with proper split paths
# Roboflow datasets: /content/datasets/train, /content/datasets/valid
data_yaml = {
    'path': '/content/datasets',      # Root directory of your extracted data
    'train': 'train/images',          # Training images
    'val': 'valid/images',            # Validation images
    'test': 'test/images',            # Test images 
    'nc': 5,                          # Number of classes
    'names': ['helmet', 'no-helmet', 'no-vest', 'person', 'vest'] #PPE classes
}

# Write YAML file
with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("--- Professional data.yaml Created ---")
with open('/content/data.yaml', 'r') as f:
    print(f.read())

**Step 4: Start Training**

explicitly tell YOLO to use the GPU

In [ ]:
# Step 4: Load the 2026 base model
model = YOLO("yolo11n.pt")

# Train the model
# epochs: How many times to study the data (50-100 is good for 100 images)
# imgsz: Standard size is 640 pixels
# Optional: Explicitly tell it to use the T4 (Device 0)
results = model.train(
    data="/content/data.yaml",
    model="yolo11n.pt", # Use YOLO11 for 2026 performance/speed
    epochs=100,
    imgsz=640,
    device=0,           # T4 GPU
    batch=16,
    augment=True,
    degrees=15.0,       # Increased: Factory cameras are often tilted
    mosaic=1.0,         # Mixes 4 images into 1 to help detect small objects
    mixup=0.1,          # Helps the model handle overlapping workers
    patience=50         # Early stopping: if it stops improving for 50 epochs, it saves time
)

**Step 5: Test on a New Image**

Upload a random picture of your calculator to the sidebar (e.g., `test_calc.jpg`) and run this:

In [ ]:
# Load PPE model
my_model = YOLO("/content/runs/detect/train/weights/best.pt")

# Test 
results = my_model.predict(source="/content/factory.png", conf=0.25, iou=0.8)

# Logic Gate Test
for r in results:
    classes_detected = [new_model.names[int(c)] for c in r.boxes.cls]

    # Check for direct violations
    is_violating = 'no-vest' in classes_detected or 'no-helmet' in classes_detected

    # Check for safety gear
    is_safe = 'vest' in classes_detected and 'helmet' in classes_detected

    if is_violating:
        print("VIOLATION DETECTED: Triggering RAG and Agents...")
    elif is_safe:
        print("COMPLIANT: All gear detected.")
    else:
        print("Scene analyzed: No clear safety status determined.")

# Display result
r.show()

In [ ]:
new_model = YOLO("/content/runs/detect/train/weights/best.pt")
print(f"Model Classes: {new_model.names}")
# It MUST show: {0: 'helmet', 1: 'no-helmet', 2: 'no-vest', 3: 'person', 4: 'vest'}